In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run with 2 epochs per network; grading always uses it.
# False: training for EPOCHS epochs (the data is generated, no download).
IS_UNITTEST = True
# Epochs of each of the three networks when IS_UNITTEST = False (about
# 15 seconds per network for 60 epochs on a laptop CPU).
EPOCHS = 60
# Device of the full run. 'auto' (the default) takes the first available of
# 'cuda' (NVIDIA GPU), 'xla' (TPU), 'mps' (Apple GPU, only when Python runs on
# macOS itself) and 'cpu'; the cell prints the one it took. The small run
# always uses the CPU.
DEVICE = 'auto'
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)

# -- start --
data_dir = course_utils.checker.get_data_dir()
course_utils.checker.save_npz({'seed': np.array(0)}, data_dir / 'inputs_public.npz')
course_utils.checker.save_npz({'seed': np.array(42)}, data_dir / 'inputs_private.npz')
if not course_utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(data_dir / 'inputs_public.npz', 'xinputs.npz')
# -- end --

seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
# -- start --
# The teacher's notebook runs with the teacher's copy of the shared code.
if course_utils.hidden.detect():
    course_utils.hidden.replace_code()
# -- end --
src = source.code.src
answer = {}

# The device of the training: always the CPU in the small run, DEVICE otherwise.
device = src.utils.device.select_device('cpu' if IS_UNITTEST else DEVICE)
print(f"Training device: {src.utils.device.describe(device)}"
      + (" (the small run always uses the CPU)" if IS_UNITTEST else f" (DEVICE = {DEVICE!r})"))


# Normalization of Features and Labels

Images come with pixels on one scale. Tables do not: an apartment has an
area of 30–300 m², 1–8 rooms, a year of construction about 2000, and a
price of about $10^5$–$10^6$. Gradient descent takes one learning rate for
all weights, and on such data it fails twice:

- **features**: the gradient of a weight is proportional to its input
  (for $y = w x$, $\partial L / \partial w = x\, \partial L / \partial y$).
  The weight of the year gets gradients 2000 times larger than that of
  the number of rooms; a learning rate small enough for the one is far too
  small for the other, and a learning rate fit for the other overflows;
- **labels**: with MSE, $\partial L / \partial \hat y = 2(\hat y - y)$; for
  prices of $10^5$ the errors and all gradients are about $10^5$ times
  larger than for targets of size 1.

The cure is the same for both: **standardize** every column to mean 0 and
standard deviation 1 with the statistics of the training set, train in
these units, and turn the predictions back into prices at the end.

**Files to edit:** [src/datasets/house_prices.py](../../source/code/src/datasets/house_prices.py) and the
marked cell of this notebook.


In [ ]:
src.utils.seed.seed_all(0)


# The Data

`make_house_prices` (given) generates houses with 5 features and a price
that depends on them in a non-linear way, plus noise. 5 000 houses for
training, 1 000 for validation.


In [ ]:
features, prices = src.datasets.house_prices.make_house_prices(6000, seed=seed)
train_features, valid_features = features[:5000], features[5000:]
train_prices, valid_prices = prices[:5000], prices[5000:]

for name, column in zip(src.datasets.house_prices.FEATURE_NAMES, train_features.T):
    print(f'{name:24s} min {column.min():8.1f}  max {column.max():8.1f}  std {column.std():7.1f}')
print(f"{'price':24s} min {train_prices.min():8.0f}  max {train_prices.max():8.0f}  std {train_prices.std():7.0f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(train_features[:, 0], train_prices, s=2)
axes[0].set_xlabel('area, m2')
axes[1].scatter(train_features[:, 3], train_prices, s=2)
axes[1].set_xlabel('distance to centre, km')
for ax in axes:
    ax.set_ylabel('price')
plt.show()


# Task 1: The Standardizer

## Goal

Implement `Standardizer` in [src/datasets/house_prices.py](../../source/code/src/datasets/house_prices.py):

- `fit(values)`: remember `mean` and `std` of `values` over the first
  dimension (one number per column for `[n, 5]`, one number for `[n]`);
  return `self`;
- `transform(values)`: $(x - \text{mean}) / \text{std}$;
- `inverse_transform(values)`: $x \cdot \text{std} + \text{mean}$.

A new `Standardizer()` that was never fitted has mean 0 and std 1, so it
changes nothing; we use that for the runs on raw data.


In [ ]:
# Edit: ../../source/code/src/datasets/house_prices.py
feature_scaler = src.datasets.house_prices.Standardizer().fit(train_features)
price_scaler = src.datasets.house_prices.Standardizer().fit(train_prices)
# ^^^ This class should be implemented in this task

scaled = feature_scaler.transform(train_features)
answer['standardizer'] = {
    'feature_mean': feature_scaler.mean,
    'feature_std': feature_scaler.std,
    'price_mean': price_scaler.mean,
    'price_std': price_scaler.std,
    'scaled_rows': scaled[:3],
    'restored_prices': price_scaler.inverse_transform(price_scaler.transform(train_prices[:3])),
}
print('scaled features: mean', scaled.mean(dim=0), 'std', scaled.std(dim=0))


In [ ]:
course_utils.checker.print_progress(
    'Task 1: The Standardizer', answer,
    prefixes=('standardizer',),
    is_small_run=IS_UNITTEST)


# Task 2: The Loss and the Metric

## Goal

`make_loss_and_metric(price_scaler)` returns two functions of the batch
protocol. The network (`Regressor`, given: 5 → 64 → 64 → 1 with ReLU)
writes its prediction to `batch['signals']['output']` (`[N]`) **in the
units of `price_scaler`**; the true prices are `batch['data']['price']`.

- `loss`: the MSE in these units,
  $L = \frac1N \sum_i \big(\hat z_i - \mathrm{transform}(y_i)\big)^2$;
- `squared_error`: the squared error **in prices**, numerator
  $\sum_i \big(\mathrm{inverse\_transform}(\hat z_i) - y_i\big)^2$,
  denominator $N$. `train_model` divides the sums, and the square root of
  the result is the RMSE (root mean squared error) in prices: comparable
  between networks trained in different units.


In [ ]:
# Edit this cell in task.ipynb.
def make_loss_and_metric(price_scaler):
    def loss(batch):
        return batch['signals']['output'].abs().sum()

    def squared_error(batch):
        return 0, 1

    ## YOUR CODE HERE
    # -- start --
    def loss(batch):
        target = price_scaler.transform(batch['data']['price'])
        return F.mse_loss(batch['signals']['output'], target)

    def squared_error(batch):
        predicted_prices = price_scaler.inverse_transform(batch['signals']['output'])
        return (predicted_prices - batch['data']['price']).pow(2).sum(), len(batch['data']['price'])
    # -- end --

    return loss, squared_error


check_loss, check_metric = make_loss_and_metric(price_scaler)
scored_batch = {'signals': {'output': torch.tensor([0.5, -1.0, 2.0])},
                'data': {'price': train_prices[:3]}}
answer['test_loss_val'] = check_loss(scored_batch)
answer['test_metrics_val'] = check_metric(scored_batch)
print('loss', answer['test_loss_val'], 'squared error', answer['test_metrics_val'])


In [ ]:
course_utils.checker.print_progress(
    'Task 2: The Loss and the Metric', answer,
    prefixes=('test_loss_val', 'test_metrics_val'),
    is_small_run=IS_UNITTEST)


# The Experiment

The same network, the same SGD (`lr=0.01`, momentum 0.9, batches of 50),
three ways to feed it:

| run | features | prices |
|---|---|---|
| raw features | raw | standardized |
| raw prices | standardized | raw |
| both standardized | standardized | standardized |


In [ ]:
raw = src.datasets.house_prices.Standardizer()
runs = {
    'raw features': (raw, price_scaler),
    'raw prices': (feature_scaler, raw),
    'both standardized': (feature_scaler, price_scaler),
}
n_epochs = 2 if IS_UNITTEST else EPOCHS
rmse = {}
networks = {}
for name, (run_feature_scaler, run_price_scaler) in runs.items():
    train_set = src.datasets.house_prices.HousePriceDataset(run_feature_scaler.transform(train_features), train_prices)
    valid_set = src.datasets.house_prices.HousePriceDataset(run_feature_scaler.transform(valid_features), valid_prices)
    train_dl = torch.utils.data.DataLoader(train_set, batch_size=50, shuffle=True,
                                           generator=torch.Generator().manual_seed(0))
    valid_dl = torch.utils.data.DataLoader(valid_set, batch_size=1000, shuffle=False)
    run_loss, run_metric = make_loss_and_metric(run_price_scaler)

    src.utils.seed.seed_all(0)
    network = src.models.feedforward.regression.Regressor(dim_input=5)
    optimizer = torch.optim.SGD(network.parameters(), lr=0.01, momentum=0.9)
    history = src.train_loop.train_model(
        network, n_epochs, train_dl, valid_dl, {'mse': run_loss}, optimizer,
        metrics={'squared_error': run_metric}, run_name=name,
        device=device)
    rmse[name] = np.sqrt(history['valid_metrics']['squared_error'])
    networks[name] = (network, run_feature_scaler, run_price_scaler)

# The grader does not accept NaN: a diverged value is saved as -1.
answer['experiments'] = {name: np.nan_to_num(values, nan=-1.0, posinf=-1.0, neginf=-1.0)
                         for name, values in rmse.items()}

plt.figure(figsize=(8, 4))
for name, values in rmse.items():
    label = name if np.all(np.isfinite(values)) else f'{name}: diverged (NaN), no line'
    plt.plot(range(1, len(values) + 1), values, 'o-', markersize=3, label=label)
plt.axhline(float(valid_prices.std()), color='gray', linestyle=':', label='std of the prices (predicting the mean)')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('validation RMSE, price')
plt.legend()
plt.show()
for name, values in rmse.items():
    print(f'{name:18s} final validation RMSE: {values[-1]:,.0f}' if np.isfinite(values[-1])
          else f'{name:18s} diverged (NaN) in epoch {int(np.argmax(~np.isfinite(values))) + 1}')


In [ ]:
course_utils.checker.print_progress(
    'The Experiment', answer,
    prefixes=('experiments',),
    is_small_run=IS_UNITTEST)


# The Predictions

Prices predicted by the network trained on standardized data against the
true prices of the validation houses; a perfect model puts every point on
the diagonal.


In [ ]:
network, run_feature_scaler, run_price_scaler = networks['both standardized']
network.eval()
with torch.no_grad():
    predicted = run_price_scaler.inverse_transform(
        network({'data': {'features': run_feature_scaler.transform(valid_features)}})['signals']['output'])
plt.figure(figsize=(4.5, 4.5))
plt.scatter(valid_prices, predicted, s=3)
limits = [float(valid_prices.min()), float(valid_prices.max())]
plt.plot(limits, limits, color='gray')
plt.xlabel('true price')
plt.ylabel('predicted price')
plt.show()
r2 = 1 - float((predicted - valid_prices).pow(2).mean() / valid_prices.var())
print(f'R2 on the validation houses: {r2:.3f}')


What to look for (full run, 60 epochs, about 45 seconds):

- **raw features** diverge in the first epoch: the year (about 2000) makes
  the first layer's outputs and gradients huge, and one SGD step throws the
  weights to infinity;
- **raw prices** diverge as well: the error of the first predictions is
  about $10^5$, its gradient is $10^5$ times larger than the network can
  take at this learning rate;
- **both standardized** learn: the RMSE falls to about 21 000, close to the
  noise of 20 000 in the data, and $R^2$ is about 0.98.

A smaller learning rate does not save the raw runs: small enough for the
largest gradients, it is far too small for the rest. Adaptive optimizers
like Adam rescale every weight's step and survive raw data, but still
learn worse (in a test, $R^2$ about 0.9 against 0.98).


In [ ]:
course_utils.checker.save_answer(answer)
